<a href="https://colab.research.google.com/github/amal-joji/nlp/blob/main/Unit5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from transformers import pipeline

def run_chatbot():
    # The 'conversational' task is not directly supported by the pipeline function in this version.
    # Using 'text-generation' with a conversational model as an alternative.
    # 'microsoft/DialoGPT-medium' is a good model for conversational text generation.
    chatbot = pipeline("text-generation", model="microsoft/DialoGPT-medium", max_new_tokens=50)

    # For text-generation, input is typically a string.
    user_input = "Hello, how are you?"

    # Generate a response. The output is a list of dictionaries.
    result = chatbot(user_input)
    chatbot_response = result[0]['generated_text']

    print("User Input:", user_input)
    print("Chatbot Response:", chatbot_response)

if __name__ == "__main__":
    run_chatbot()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User Input: Hello, how are you?
Chatbot Response: Hello, how are you?


In [5]:
from transformers import pipeline

def run_qa():
    qa_pipeline = pipeline("question-answering")

    context = """
    India is a country in Asia. The capital of India is New Delhi.
    It is known for its rich cultural heritage.
    """

    question = "What is the capital of India?"

    result = qa_pipeline(question=question, context=context)

    print("Question:", question)
    print("Answer:", result['answer'])
    print("Confidence Score:", result['score'])

if __name__ == "__main__":
    run_qa()

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Question: What is the capital of India?
Answer: New Delhi
Confidence Score: 0.9901931285858154


In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def run_summarization():
    # The 'summarization' task is not recognized, and T5 models are not supported for 'text-generation' pipeline.
    # To properly perform summarization with t5-small, we will load the tokenizer and model directly.
    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

    text = """
    Artificial Intelligence (AI) is transforming industries by automating tasks,
    improving efficiency, and enabling smarter decision-making.
    It is widely used in healthcare, finance, education, and more.
    """

    # T5 models often require a prefix for summarization tasks.
    input_text = "summarize: " + text

    # Tokenize the input text
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

    # Generate the summary
    # We set max_length and min_length for controlling summary length.
    # num_beams > 1 for beam search, which often yields better quality.
    # early_stopping=True stops generation once all beam hypotheses have finished.
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=40,
        min_length=10,
        num_beams=4,
        early_stopping=True
    )

    # Decode the generated summary IDs back to text
    summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    print("Original Text:\n", text)
    print("\nGenerated Summary:\n", summary_text)

if __name__ == "__main__":
    run_summarization()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Original Text:
 
    Artificial Intelligence (AI) is transforming industries by automating tasks,
    improving efficiency, and enabling smarter decision-making.
    It is widely used in healthcare, finance, education, and more.
    

Generated Summary:
 artificial intelligence (AI) is transforming industries by automating tasks. it is widely used in healthcare, finance, education, and more.


In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def run_translation():
    # The 'translation' task is not directly recognized by the pipeline, even with a specific model.
    # To properly perform translation, we will load the tokenizer and model directly.
    tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
    model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-fr")

    text = "Hello, how are you?"

    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt")

    # Generate the translation
    translated_ids = model.generate(**inputs)

    # Decode the generated IDs back to text
    translated_text = tokenizer.decode(translated_ids[0], skip_special_tokens=True)

    print("Original:", text)
    print("Translated:", translated_text)

if __name__ == "__main__":
    run_translation()

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Original: Hello, how are you?
Translated: Bonjour, comment allez-vous ?
